# Lab 6 · Propagación de fallos y resiliencia de la red
### Tema 6 · MU Gestión de Sistemas Complejos · Caso transversal: operador de telecomunicaciones

**Entorno:** SageMaker Notebook (kernel Python 3) · **Librerías:** pandas, NetworkX, matplotlib
**Duración estimada:** 180 min · **Nivel:** intermedio

---

## 1. Idea general del laboratorio

Este laboratorio se construye **sobre la red del Lab 5**. Allí medimos qué nodos eran críticos; aquí
**ponemos esa red a prueba**: ¿qué ocurre cuando un nodo cae? ¿El fallo se contiene o se propaga? ¿Cuántos
clientes quedan aislados? La resiliencia es una **propiedad emergente**: no reside en ningún componente
aislado, sino en cómo están conectados. Simularemos caídas (aleatorias y dirigidas), mediremos la
degradación de la conectividad y la cobertura de clientes, y propondremos refuerzos **simulándolos** antes
de invertir. **No se entrena ningún modelo**: se simula el sistema bajo estrés.

### Objetivos didácticos
- Explicar la resiliencia como propiedad estructural.
- Simular caídas aleatorias y dirigidas (ataque a los más centrales).
- Medir la degradación con la componente conexa mayor y los clientes conectados.
- Comparar robustez frente a azar vs ataque dirigido.
- Proponer y **evaluar** medidas de refuerzo simulándolas.

> **Nota para el vídeo.** Cada celda está comentada paso a paso. Reutilizamos los CSV del Lab 5, incluidos
> en este paquete.

In [ ]:
# ===========================================================================
# CELDA 1 · Dependencias
# ===========================================================================
import sys
!{sys.executable} -m pip install -q networkx matplotlib pandas
print("Dependencias listas")

## Parte A · Reconstruir el grafo

Igual que en el Lab 5: nodos + aristas + clientes como atributo. Es el punto de partida de las simulaciones.

In [ ]:
# ===========================================================================
# CELDA 2 · Cargar datos y reconstruir el grafo con clientes
# ===========================================================================
import pandas as pd, networkx as nx, matplotlib.pyplot as plt, random
import numpy as np

# RUTA local por defecto; para S3 usa 's3://<tu-bucket>/graph'.
RUTA = '.'
nodes = pd.read_csv(f'{RUTA}/network_nodes.csv')
edges = pd.read_csv(f'{RUTA}/network_edges.csv')
cust  = pd.read_csv(f'{RUTA}/customer_nodes.csv')

# Grafo no dirigido con tipo y region por nodo.
G = nx.Graph()
for _, r in nodes.iterrows():
    G.add_node(r['node_id'], node_type=r['node_type'], region=r['region'])
for _, r in edges.iterrows():
    G.add_edge(r['source'], r['target'])

# Clientes por nodo de acceso -> atributo 'n_clientes' (0 en core/aggregation).
cpn = cust.groupby('access_node_id').size()
for n in G.nodes():
    G.nodes[n]['n_clientes'] = int(cpn.get(n, 0))

# Total de clientes: lo usaremos para expresar la cobertura en porcentaje.
TOTAL_CLIENTES = int(cust.shape[0])
print("Nodos:", G.number_of_nodes(), "Clientes:", TOTAL_CLIENTES)

### Función de medida: clientes conectados al núcleo

Un cliente tiene servicio si **existe un camino** desde su acceso hasta algún nodo de núcleo. Es el corazón
de todas las métricas de cobertura.

In [ ]:
# ===========================================================================
# CELDA 3 · Funcion: clientes que AUN tienen servicio (camino al nucleo)
# ===========================================================================
def clientes_conectados(H):
    # Lista de nodos de nucleo que SIGUEN en el grafo H (puede que ya cayeran).
    cores = [n for n in H.nodes() if H.nodes[n]["node_type"] == "core"]
    if not cores:                 # si no queda ningun nucleo, nadie tiene servicio
        return 0
    conectados = 0
    for n, a in H.nodes(data=True):
        if a["n_clientes"] > 0:                       # solo nodos con clientes
            # any(...) = True si el nodo alcanza AL MENOS un nucleo.
            if any(nx.has_path(H, n, c) for c in cores):
                conectados += a["n_clientes"]
    return conectados

# Comprobacion inicial: con la red intacta deberian estar todos conectados.
print("Clientes conectados al inicio:", clientes_conectados(G))

## Parte B · Simulación de fallos aleatorios

Quitamos nodos al azar, uno a uno, y medimos la degradación. Como el azar varía, **repetimos 20 veces y
promediamos** para una curva estable.

In [ ]:
# ===========================================================================
# CELDA 4 · Funcion generica de simulacion de caidas
# ===========================================================================
def simular_fallos(G, orden, total_clientes):
    # H = copia que iremos vaciando segun 'orden' (lista de nodos a eliminar).
    H = G.copy()
    serie_clientes, serie_componente = [], []
    for nodo in orden:
        if nodo in H:
            H.remove_node(nodo)          # cae el nodo
        # 1) fraccion de clientes que aun tienen servicio
        serie_clientes.append(clientes_conectados(H) / total_clientes)
        # 2) tamano de la mayor pieza conexa, como fraccion del total de nodos
        if H.number_of_nodes() > 0:
            mayor = max(nx.connected_components(H), key=len)  # mayor componente
            serie_componente.append(len(mayor) / G.number_of_nodes())
        else:
            serie_componente.append(0)
    return serie_clientes, serie_componente

# --- Promedio de 20 ordenes aleatorios ---
rep = 20
acc_cli = np.zeros(G.number_of_nodes())   # acumulador de las 20 curvas
for _ in range(rep):
    orden = list(G.nodes()); random.shuffle(orden)   # orden de caida aleatorio
    cli, _ = simular_fallos(G, orden, TOTAL_CLIENTES)
    acc_cli += np.array(cli)
aleatorio = acc_cli / rep                 # media de las 20 repeticiones
print("Curva aleatoria calculada (media de %d repeticiones)" % rep)

## Parte C · Simulación de ataque dirigido

Ahora quitamos los nodos en **orden de criticidad** (mayor intermediación primero).

In [ ]:
# ===========================================================================
# CELDA 5 · Ataque dirigido: caen primero los nodos mas centrales
# ===========================================================================
# Intermediacion de cada nodo (igual que en el Lab 5).
bet = nx.betweenness_centrality(G)
# Ordenamos los nodos de MAYOR a menor intermediacion: ese es el orden de ataque.
orden_dirigido = sorted(G.nodes(), key=lambda n: bet[n], reverse=True)
# Simulamos una sola pasada (el ataque es determinista, no hace falta promediar).
dirigido, comp_dirigido = simular_fallos(G, orden_dirigido, TOTAL_CLIENTES)
print("Primeros nodos atacados:", orden_dirigido[:6])

In [ ]:
# ===========================================================================
# CELDA 6 · Comparacion grafica: azar vs ataque dirigido
# ===========================================================================
plt.figure(figsize=(10, 6))
# Eje X: porcentaje de nodos caidos (de 1 nodo hasta todos).
x = np.arange(1, G.number_of_nodes() + 1) / G.number_of_nodes() * 100
# Multiplicamos por 100 para expresar la cobertura en %.
plt.plot(x, aleatorio * 100, label="Fallo aleatorio (media de 20)", lw=2)
plt.plot(x, np.array(dirigido) * 100, label="Ataque dirigido", lw=2)
plt.xlabel("% de nodos caidos")
plt.ylabel("% de clientes aun con servicio")
plt.title("Resiliencia: fallo aleatorio vs ataque dirigido")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

> **Atención (vídeo).** La curva dirigida cae **mucho más rápido**. Esa brecha mide la **fragilidad
> estructural**: la red aguanta los golpes fortuitos pero es muy sensible a perder sus nodos centrales.

## Parte D · Vulnerabilidades concretas

### D.1 · ¿Cuántos nodos hay que quitar para partir la red?

In [ ]:
# ===========================================================================
# CELDA 7 · Umbral de fragmentacion bajo ataque dirigido
# ===========================================================================
H = G.copy()
quitados = 0
for nodo in orden_dirigido:
    H.remove_node(nodo); quitados += 1
    if H.number_of_nodes() == 0:
        break
    # Mayor pieza conexa tras esta caida.
    mayor = max(nx.connected_components(H), key=len)
    # En cuanto la mayor pieza baja del 50% de los nodos, la red esta "partida".
    if len(mayor) < 0.5 * G.number_of_nodes():
        print(f"Con {quitados} nodos dirigidos cae bajo el 50% de conectividad")
        print("Nodos atacados hasta ahi:", orden_dirigido[:quitados])
        break

### D.2 · Impacto de cada caída individual

In [ ]:
# ===========================================================================
# CELDA 8 · Clientes que aisla la caida de cada nodo individual
# ===========================================================================
impacto = {}
base = clientes_conectados(G)            # clientes conectados con la red intacta
for n in G.nodes():
    H = G.copy(); H.remove_node(n)       # quitamos un solo nodo
    # Diferencia = cuantos clientes pierden el servicio por su caida.
    impacto[n] = base - clientes_conectados(H)

imp = pd.Series(impacto).sort_values(ascending=False)
print("Nodos que mas clientes aislan si caen (top 10):")
print(imp.head(10))

**Preguntas.** ¿Coinciden con los puntos de articulación del Lab 5 (las agregaciones periféricas)? Esos son
los candidatos número uno a reforzar.

## Parte E · Simular medidas de refuerzo

La simulación permite **evaluar soluciones antes de invertir**. Añadimos un anillo de redundancia entre
agregaciones y volvemos a medir.

In [ ]:
# ===========================================================================
# CELDA 9 · Anadir redundancia (anillo entre agregaciones) y re-evaluar
# ===========================================================================
G2 = G.copy()                            # red reforzada = copia + enlaces nuevos
# Lista de nodos de agregacion.
agg = [n for n in G2.nodes() if G2.nodes[n]["node_type"] == "aggregation"]
# Conectamos cada agregacion con la siguiente en circulo (anillo): un enlace
# extra que da rutas alternativas si cae un nodo. El modulo % cierra el anillo.
for i in range(len(agg)):
    G2.add_edge(agg[i], agg[(i + 1) % len(agg)])

# Recalculamos intermediacion y volvemos a atacar la red reforzada.
bet2 = nx.betweenness_centrality(G2)
orden2 = sorted(G2.nodes(), key=lambda n: bet2[n], reverse=True)
dirigido2, _ = simular_fallos(G2, orden2, TOTAL_CLIENTES)

# Comparamos la curva de ataque original con la de la red reforzada.
plt.figure(figsize=(10, 6))
plt.plot(x, np.array(dirigido) * 100, label="Red original (ataque)", lw=2)
plt.plot(x, np.array(dirigido2) * 100, label="Red reforzada (ataque)", lw=2)
plt.xlabel("% de nodos caidos"); plt.ylabel("% de clientes con servicio")
plt.title("Efecto del refuerzo de redundancia")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ===========================================================================
# CELDA 10 · ¿Desaparecen puntos de articulacion con el refuerzo?
# ===========================================================================
# Si el anillo elimina cuellos, habra menos (o ningun) punto de articulacion.
print("Articulacion original:", list(nx.articulation_points(G)))
print("Articulacion reforzada:", list(nx.articulation_points(G2)))

**Preguntas.** ¿Mejora la red reforzada su resistencia? ¿Desaparecen puntos de articulación? Si solo
pudieras añadir **dos** enlaces, ¿dónde los pondrías?

> **Buena práctica.** Simular el efecto de una inversión antes de hacerla convierte decisiones de millones
> en experimentos de minutos.

## Parte F · Entregable · Parte G · Cierre

Entregable (5–7 págs.): experimento azar vs dirigido y su gráfica; nodos dirigidos para bajar del 50 %;
top-10 de impacto individual; resultado del refuerzo y su rentabilidad; recomendación (qué reforzar
primero, ligado al Lab 5); y reflexión sobre por qué una red jerárquica es robusta ante el azar pero
frágil ante ataques. Al terminar, **Stop** la instancia y **conserva los datos: el Lab 7 vuelve a usar esta
red** (más un fichero de comportamiento).

---

### Cierre didáctico
Robustez y fragilidad conviven en la misma estructura, según el tipo de golpe. El Lab 7 cerrará el bloque
de redes aplicando **IA sobre el grafo** para detectar anomalías.